# Notebook 5 — Production Practices & Industry Insights

Notebooks 1–4 were about **correctness**: invoke the agent, read the trace, build the loop by hand, control loops, sampling, and hallucinations. This notebook is about **operations** — the gap between "it worked in my notebook" and "it runs in production without paging me at 2 a.m."

Everything here is on the same **TravelMind** agent and the same `us-east-1` access you used in Notebook 1. Nothing new to install.

**Run context (same as Notebook 1):**

- *VS Code*: (1) activate your `.venv`, (2) confirm `aws configure` creds + `AWS_DEFAULT_REGION=us-east-1`, (3) select the `.venv` kernel.
- *Google Colab*: (1) `!pip install boto3`, (2) set creds via Colab **Secrets** or `os.environ` (never paste keys in a cell), (3) keep `region_name="us-east-1"`.

What you will build: env-driven config, a least-privilege IAM policy, a retry-hardened client, structured logging, cost controls, production aliases, an eval harness, and input/PII guards. Each section ends with the one-line principle.

## 1. Config from scratch — environment, not hardcoding

The fastest way to leak credentials is to type them into a cell. The second fastest is to hardcode an account id or a region and ship it. Production config comes from the **environment**, with a **fail-fast** check so a missing value stops you on line one instead of three API calls later.

The pattern below reads everything from `os.environ`, falls back to safe defaults only for non-secret values, and refuses to continue if the agent id is still a placeholder.

In [ ]:
import os, boto3
from botocore.exceptions import ClientError

# --- non-secret config: environment first, sensible fallback second ---
REGION   = os.environ.get("AWS_DEFAULT_REGION", "us-east-1")
AGENT_ID = os.environ.get("TRAVELMIND_AGENT_ID", "XXXXXXXXXX")   # <-- export this, or paste once below
ALIAS_ID = os.environ.get("TRAVELMIND_ALIAS_ID", "TSTALIASID")   # dev alias; section 7 replaces it

# --- secrets are NEVER read here: boto3 picks them up from the environment / role automatically ---
# AWS_ACCESS_KEY_ID, AWS_SECRET_ACCESS_KEY, AWS_SESSION_TOKEN  (env or role)  -> handled by boto3

# fail fast: a missing/placeholder id should stop you now, not mid-demo
problems = []
if AGENT_ID == "XXXXXXXXXX":
    problems.append("AGENT_ID is still the placeholder — export TRAVELMIND_AGENT_ID or set it here.")
if REGION != "us-east-1":
    problems.append(f"REGION is {REGION!r}; this material is scoped to us-east-1.")
assert not problems, "CONFIG ERROR:\n- " + "\n- ".join(problems)

NOVA_MODEL   = "amazon.nova-lite-v1:0"                          # cheap, on-demand
CLAUDE_MODEL = "us.anthropic.claude-3-5-haiku-20241022-v1:0"    # cross-region 'us.' profile

sts = boto3.client("sts", region_name=REGION)
ACCOUNT_ID = sts.get_caller_identity()["Account"]
AGENT_ARN       = f"arn:aws:bedrock:{REGION}:{ACCOUNT_ID}:agent/{AGENT_ID}"
AGENT_ALIAS_ARN = f"arn:aws:bedrock:{REGION}:{ACCOUNT_ID}:agent-alias/{AGENT_ID}/{ALIAS_ID}"
print("account:", ACCOUNT_ID, "| region:", REGION, "| agent:", AGENT_ID)

**Principle:** config and secrets come from the environment or the role, never from a literal in the code. Fail fast on anything missing.

## 2. IAM — roles, not access keys

An **access key** is a long-lived secret string. If it leaks (a commit, a log line, a screenshot), anyone can use it until you rotate it. A **role** issues short-lived, automatically-rotated credentials to a workload that is allowed to assume it — there is no secret to leak.

| Where your code runs | Use this identity |
|---|---|
| EC2 | instance profile (an attached role) |
| ECS / Fargate | task role |
| Lambda | execution role |
| EKS | IRSA (role for service account) |
| A developer laptop | short-lived SSO creds (`aws sso login`), not a static key |

The role is only half of it. The other half is **least privilege**: the role should be allowed to do exactly what this agent needs and nothing else. Below is the policy a TravelMind *caller* actually needs — invoke this one agent alias, and converse with the two models it uses.

In [ ]:
import json

# Least-privilege policy for a service that ONLY invokes the TravelMind agent.
# Note the resource ARNs are specific — not "*".
MODEL_ARN_NOVA    = f"arn:aws:bedrock:{REGION}::foundation-model/{NOVA_MODEL}"
PROFILE_ARN_CLAUDE = f"arn:aws:bedrock:{REGION}:{ACCOUNT_ID}:inference-profile/{CLAUDE_MODEL}"
# A cross-region ('us.') profile also needs InvokeModel on the underlying FM ARN in EACH region it routes to.
FM_ARN_CLAUDE_USE1 = f"arn:aws:bedrock:us-east-1::foundation-model/anthropic.claude-3-5-haiku-20241022-v1:0"
FM_ARN_CLAUDE_USE2 = f"arn:aws:bedrock:us-east-2::foundation-model/anthropic.claude-3-5-haiku-20241022-v1:0"

travelmind_caller_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "InvokeTheAgentAlias",
            "Effect": "Allow",
            "Action": ["bedrock:InvokeAgent"],
            "Resource": [AGENT_ALIAS_ARN],
        },
        {
            "Sid": "ConverseWithItsModels",   # Converse maps to the InvokeModel permission
            "Effect": "Allow",
            "Action": ["bedrock:InvokeModel", "bedrock:InvokeModelWithResponseStream"],
            "Resource": [MODEL_ARN_NOVA, PROFILE_ARN_CLAUDE, FM_ARN_CLAUDE_USE1, FM_ARN_CLAUDE_USE2],
        },
    ],
}
print(json.dumps(travelmind_caller_policy, indent=2))

# How you would attach it (do NOT run blindly — adapt names to your account):
#   iam = boto3.client("iam")
#   iam.put_role_policy(RoleName="travelmind-caller",
#                       PolicyName="travelmind-invoke",
#                       PolicyDocument=json.dumps(travelmind_caller_policy))

**The cross-region gotcha (why two extra ARNs):** the `us.` Claude id is an *inference profile* that routes to the foundation model in more than one region. The caller needs `InvokeModel` on the **profile ARN** *and* on the **foundation-model ARN in each region the profile can route to**. Miss the underlying-region ARNs and you get `AccessDenied` only sometimes — when traffic happens to route to the region you forgot. Confirm the exact regions your profile lists in the console catalog.

**Principle:** identity is a role, permission is least-privilege and scoped to specific ARNs. "`Action: bedrock:*` on `Resource: *`" is how you fail a security review.

## 3. Retries + exponential backoff

Bedrock is multi-tenant. Under load you will get `ThrottlingException` — it is normal, not a bug. The fix is not "ask for more quota" first; it is to **retry with backoff** so brief throttles are invisible to your user. `botocore` does this for you when you configure the client.

In [ ]:
from botocore.config import Config

# Adaptive retries: backoff + a client-side rate limiter that slows down when it sees throttling.
hardened = Config(
    region_name=REGION,
    retries={"max_attempts": 6, "mode": "adaptive"},   # modes: legacy | standard | adaptive
    connect_timeout=5,
    read_timeout=60,                                    # agent turns can be slow; do not cut them off early
)

bedrock        = boto3.client("bedrock",               config=hardened)  # control: models, guardrails, logging
bedrock_runtime= boto3.client("bedrock-runtime",       config=hardened)  # data:    converse, apply_guardrail
agent_build    = boto3.client("bedrock-agent",         config=hardened)  # control: agents, aliases, action groups
agent_runtime  = boto3.client("bedrock-agent-runtime", config=hardened)  # data:    invoke_agent

print("retry mode:", hardened.retries["mode"], "| max_attempts:", hardened.retries["max_attempts"])

In [ ]:
import time, random
from botocore.exceptions import ClientError

# Belt-and-braces: a manual backoff wrapper for the rare error botocore will not retry for you
# (e.g. a dependency you call yourself, or when you want jittered backoff with your own ceiling).
THROTTLES = {"ThrottlingException", "TooManyRequestsException", "ServiceQuotaExceededException"}

def with_backoff(fn, *args, max_tries=5, base=0.5, **kwargs):
    for attempt in range(max_tries):
        try:
            return fn(*args, **kwargs)
        except ClientError as e:
            code = e.response["Error"]["Code"]
            if code not in THROTTLES or attempt == max_tries - 1:
                raise
            sleep = base * (2 ** attempt) + random.uniform(0, 0.3)   # exponential + jitter
            print(f"  throttled ({code}); retry {attempt+1} in {sleep:.2f}s")
            time.sleep(sleep)

# usage: with_backoff(bedrock_runtime.converse, modelId=NOVA_MODEL, messages=[...])
print("with_backoff ready")

**Principle:** throttling is expected. Configure adaptive retries on every client; add jittered manual backoff only where botocore cannot help. Never retry forever — cap the attempts.

## 4. Control plane vs data plane — in production

You met the four clients in Notebook 1. The production angle is about **frequency and the hot path**:

- **Control plane** (`bedrock`, `bedrock-agent`): `create_*`, `update_agent`, `prepare_agent`, `create_agent_alias`. Rare, low-TPS, IAM-heavy. You run these at **deploy time**, in CI — not per user request.
- **Data plane** (`bedrock-runtime`, `bedrock-agent-runtime`): `converse`, `invoke_agent`. Hot path, latency-sensitive, the ones that **throttle**.

The classic production mistake is calling a control-plane op (like `prepare_agent`, or re-creating a guardrail) inside the request handler. It is slow, it has tight limits, and it does not belong there. Resolve agent/guardrail/alias ids **once at startup**, cache them, and only touch the data plane per request.

In [ ]:
# A tiny "resolve once, reuse" cache — the shape your app startup should follow.
_CONFIG_CACHE = {}

def resolved_ids():
    if not _CONFIG_CACHE:                       # populated exactly once
        _CONFIG_CACHE.update({
            "agent_id":  AGENT_ID,
            "alias_id":  ALIAS_ID,
            "agent_arn": AGENT_ARN,
            # "guardrail_id": ...,  "guardrail_version": ...,   # add yours from Notebook 4
        })
        print("resolved ids at startup (control plane touched once)")
    return _CONFIG_CACHE

ids = resolved_ids()
ids2 = resolved_ids()        # second call hits the cache, no control-plane work
print(ids2["agent_id"], "| cache reused:", ids is ids2)

**Principle:** control-plane at deploy time, data-plane per request. Resolve ids once, cache them, never `prepare_agent` on the hot path.

## 5. Observability — the logs you emit

When an agent misbehaves in production you cannot re-run it in a notebook. You have only what you logged. So log **structured records** (one JSON object per turn) with the fields you will actually query: a request id, the session id, latency, the model, token usage, and the outcome. Plain `print` is not observability — a JSON line per turn that lands in CloudWatch is.

In [ ]:
import logging, json, time, uuid

logger = logging.getLogger("travelmind")
if not logger.handlers:                     # avoid duplicate handlers on re-run
    h = logging.StreamHandler()
    h.setFormatter(logging.Formatter("%(message)s"))   # we emit JSON ourselves
    logger.addHandler(h)
logger.setLevel(logging.INFO)

def log_event(**fields):
    fields.setdefault("ts", time.time())
    logger.info(json.dumps(fields, default=str))    # one structured line -> CloudWatch

def converse_logged(model_id, prompt, session_id, **kw):
    req = str(uuid.uuid4())[:8]
    t0 = time.time()
    try:
        resp = bedrock_runtime.converse(
            modelId=model_id,
            messages=[{"role": "user", "content": [{"text": prompt}]}],
            inferenceConfig={"temperature": kw.get("temperature", 0.0),
                             "maxTokens": kw.get("max_tokens", 256)},
        )
        usage = resp.get("usage", {})
        log_event(req=req, session=session_id, model=model_id,
                  latency_ms=round((time.time()-t0)*1000),
                  stop=resp.get("stopReason"),
                  in_tok=usage.get("inputTokens"), out_tok=usage.get("outputTokens"),
                  status="ok")
        return resp["output"]["message"]["content"][0]["text"]
    except ClientError as e:
        log_event(req=req, session=session_id, model=model_id,
                  latency_ms=round((time.time()-t0)*1000),
                  status="error", error=e.response["Error"]["Code"])
        raise

print(converse_logged(NOVA_MODEL, "In one sentence, what is a PNR?", session_id="demo-001"))

For an **agent** invoke, per-call token usage lives inside the trace metadata (`orchestrationTrace -> modelInvocationOutput -> metadata -> usage`) rather than a top-level `usage` field. Capture latency and the final-response presence the same way, and pull token counts from the trace when you need cost attribution.

Beyond your own logs, Bedrock can log the **raw model invocations** itself (input/output) to CloudWatch and/or S3 at the account level. This is an audit/debugging complement, not a replacement for app logs.

In [ ]:
# Account-level model invocation logging. Guarded so it does NOT run by default:
# it requires a pre-created CloudWatch log group (and an IAM role Bedrock can assume).
ENABLE_BEDROCK_LOGGING = False   # flip to True only after you create the log group + role

if ENABLE_BEDROCK_LOGGING:
    bedrock.put_model_invocation_logging_configuration(
        loggingConfig={
            "cloudWatchConfig": {
                "logGroupName": "/bedrock/travelmind",
                "roleArn": f"arn:aws:iam::{ACCOUNT_ID}:role/BedrockLoggingRole",
            },
            "textDataDeliveryEnabled": True,
        }
    )
    print("bedrock model-invocation logging enabled")
else:
    print("skipped (ENABLE_BEDROCK_LOGGING=False) — verify exact fields in the Bedrock docs before enabling")

**Principle:** you can only debug what you logged. Emit one structured JSON record per turn (ids, latency, tokens, outcome) and ship it to CloudWatch. Turn on Bedrock's own invocation logging for an independent audit copy.

## 6. Cost controls

Bedrock bills per **token** (input + output) and per **guardrail unit**. Three levers keep the bill predictable, and one model-selection habit cuts it hardest:

- **Cap the loop**: `max_turns` (Notebook 2) — the single biggest protection against a runaway bill.
- **Cap the output**: `maxTokens` — a verbose model on a tight task is wasted spend.
- **Tier the model**: cheap model for extraction/routing, expensive model only for hard reasoning.
- **Tune `applyGuardrailInterval`**: checking every token costs more than checking in larger chunks.

In [ ]:
# Model tiering: route the easy 80% to the cheap model, keep the strong model for hard turns.
def pick_model(task: str) -> str:
    cheap_tasks = {"extract", "classify", "route", "format", "lookup"}
    return NOVA_MODEL if task in cheap_tasks else CLAUDE_MODEL

# Illustrative cost estimate. RATES ARE PLACEHOLDERS — fill from the current Bedrock pricing page.
# Do not treat these numbers as real prices.
PRICE_PER_1K = {                    # USD per 1,000 tokens  (REPLACE with live values)
    NOVA_MODEL:   {"in": 0.0,  "out": 0.0},
    CLAUDE_MODEL: {"in": 0.0,  "out": 0.0},
}
def est_cost(model_id, in_tok, out_tok):
    r = PRICE_PER_1K.get(model_id, {"in": 0.0, "out": 0.0})
    return (in_tok/1000)*r["in"] + (out_tok/1000)*r["out"]

print("extract ->", pick_model("extract"), "| replan ->", pick_model("replan"))
print("est cost (fill rates):", est_cost(NOVA_MODEL, 350, 80))

**Principle:** cap turns, cap tokens, tier the model, batch the guardrail. Most agent cost is avoidable spend on tasks a cheap model could have handled. (And never hardcode prices as fact — they change; read them live.)

## 7. Aliases & versions — retire TSTALIASID

`TSTALIASID` always points at the **draft** — the latest, unstable edit of your agent. That is exactly what you want while building and exactly what you must not ship. Production traffic points at a **named alias** mapped to a **numbered version**, so a bad edit to the draft cannot reach users. Promotion is: prepare the draft, create a version, point the alias at it.

In [ ]:
# Create (or reuse) a production alias. Idempotent: re-running will not crash.
def ensure_alias(name="prod"):
    try:
        a = agent_build.create_agent_alias(agentId=AGENT_ID, agentAliasName=name)
        alias_id = a["agentAlias"]["agentAliasId"]
        print("created alias", name, "->", alias_id)
    except ClientError as e:
        if e.response["Error"]["Code"] != "ConflictException":
            raise
        existing = agent_build.list_agent_aliases(agentId=AGENT_ID)["agentAliasSummaries"]
        alias_id = next(x["agentAliasId"] for x in existing if x["agentAliasName"] == name)
        print("alias", name, "already exists ->", alias_id)
    return alias_id

# PROD_ALIAS_ID = ensure_alias("prod")   # uncomment to create it in your account
# In CI, after any config change:  prepare_agent -> (new version) -> point alias at it.
print("ensure_alias ready — swap ALIAS_ID to a named alias for production traffic")

**Principle:** `TSTALIASID` is for development only. Production points a named alias at a pinned version; `prepare_agent` runs in CI after a change, never on the hot path.

## 8. Idempotency — safe re-runs and CI

In Notebook 3 every create call caught the "already exists" error so a second run did not crash. Generalise that: any setup step you run in CI should be **idempotent** — running it twice leaves the same state as running it once. Without this, a re-deploy either errors out or quietly creates duplicates.

In [ ]:
def create_or_get(create_fn, conflict_codes=("ConflictException",
                                             "EntityAlreadyExistsException",
                                             "ResourceConflictException"),
                  on_exists=None):
    """Run a create; if it already exists, fall back to the lookup instead of crashing."""
    try:
        return create_fn()
    except ClientError as e:
        if e.response["Error"]["Code"] not in conflict_codes:
            raise
        if on_exists is None:
            print("  exists already; nothing to do")
            return None
        return on_exists()

# usage:
#   create_or_get(lambda: agent_build.create_agent_action_group(...),
#                 on_exists=lambda: agent_build.list_agent_action_groups(agentId=AGENT_ID, agentVersion="DRAFT"))
print("create_or_get ready")

**Principle:** every setup step is idempotent. Re-running a deploy must be a no-op, not a crash and not a duplicate.

## 9. Guardrail as an audit trail — never promise zero

The grounding guardrail from Notebook 4 does double duty in production. Its job is to **block** ungrounded answers; its by-product is a **record** of every time the model tried to drift. Log the scores on every check and you have an audit trail a regulator or a customer-success team can actually read.

In [ ]:
def guarded_answer(answer, source, query, guardrail_id, version="DRAFT", session_id="demo"):
    resp = bedrock_runtime.apply_guardrail(
        guardrailIdentifier=guardrail_id, guardrailVersion=version, source="OUTPUT",
        content=[
            {"text": {"text": source, "qualifiers": ["grounding_source"]}},
            {"text": {"text": query,  "qualifiers": ["query"]}},
            {"text": {"text": answer}},
        ],
    )
    intervened = resp["action"] == "GUARDRAIL_INTERVENED"
    # pull grounding/relevance scores out of the assessment for the audit record
    scores = {}
    for a in resp.get("assessments", []):
        for f in a.get("contextualGroundingPolicy", {}).get("filters", []):
            scores[f["type"]] = round(f.get("score", 0.0), 3)
    log_event(kind="guardrail", session=session_id, intervened=intervened, scores=scores)
    return ("I can only answer from your booking record." if intervened else answer), scores

# GUARDRAIL_ID = "...";  guarded_answer("Your flight is 6E-999.", SOURCE, "which flight?", GUARDRAIL_ID)
print("guarded_answer ready — every check is logged with its scores")

**Principle:** the guardrail blocks *and* records. Log every score and intervention. Promise a measurable, audited reduction in hallucination — never "zero."

## 10. Eval harness — measure, don't hope

"The agent seems better now" is not a release criterion. Before you change a prompt, a model, or a threshold, run a small **fixed set of cases** and compare the grounding scores. This is the smallest useful eval: a list of `(query, source, answer, should_pass)` rows, scored by the same guardrail, reported as a pass rate you can track across releases.

In [ ]:
EVAL_CASES = [
    # query, grounding source, candidate answer, should_pass
    ("Which flights can I rebook onto?",
     "Booking 6E-203 CANCELLED. Options: 6E-415 18:40, 6E-422 21:10.",
     "You can rebook onto 6E-415 at 18:40 or 6E-422 at 21:10.", True),
    ("Which flights can I rebook onto?",
     "Booking 6E-203 CANCELLED. Options: 6E-415 18:40, 6E-422 21:10.",
     "You can take 6E-900 at 06:00 tomorrow.", False),   # invented flight
    ("Why was my flight cancelled?",
     "Booking 6E-203 CANCELLED. Reason: fog.",
     "It was cancelled due to fog.", True),
]

def run_evals(guardrail_id, version="DRAFT", threshold=0.7):
    rows, correct = [], 0
    for q, src, ans, should_pass in EVAL_CASES:
        _, scores = guarded_answer(ans, src, q, guardrail_id, version, session_id="eval")
        grounding = scores.get("GROUNDING", 1.0)
        passed = grounding >= threshold
        ok = (passed == should_pass)
        correct += ok
        rows.append((q[:28], grounding, "PASS" if passed else "BLOCK", "✓" if ok else "✗"))
    print(f"{'case':30}{'grounding':>10}{'verdict':>8}{'correct':>8}")
    for r in rows:
        print(f"{r[0]:30}{r[1]:>10}{r[2]:>8}{r[3]:>8}")
    print(f"\naccuracy: {correct}/{len(EVAL_CASES)}")

# run_evals(GUARDRAIL_ID)   # needs a real guardrail id from Notebook 4
print("run_evals ready — wire in your guardrail id to score the cases")

**Principle:** no change ships without a number. A tiny fixed eval set, scored the same way every time, turns "seems better" into a tracked pass rate.

## 11. Security — PII, input validation, and where execution happens

Three habits that separate a demo from a system handling real bookings:

- **Validate input before it reaches a tool.** A malformed PNR should be rejected by your code, not discovered by the model mid-loop.
- **Redact PII before logging.** Your structured logs are useful precisely because you query them — which means you must not fill them with passenger names and full record locators.
- **Keep sensitive execution in your VPC.** The `RETURN_CONTROL` action group from Notebook 3 runs the function in *your* code, so card numbers and personal data never leave your boundary for a Lambda you would otherwise have to lock down.

In [ ]:
import re

PNR_RE = re.compile(r"^[A-Z0-9]{6}$")          # IATA record locator: 6 alphanumerics

def valid_pnr(pnr: str) -> bool:
    return bool(PNR_RE.match(pnr or ""))

def redact(text: str) -> str:
    text = re.sub(r"\b[A-Z0-9]{6}\b", "<PNR>", text)              # record locators
    text = re.sub(r"\b[\w.+-]+@[\w-]+\.[\w.-]+\b", "<EMAIL>", text)  # emails
    return text

# guard the tool boundary
def safe_lookup(pnr):
    if not valid_pnr(pnr):
        return {"error": "invalid PNR format"}     # rejected before any tool/model call
    return {"pnr": pnr, "status": "ok"}            # real lookup would go here

print("valid 6E-203?", valid_pnr("6E-203"), "| valid ABC123?", valid_pnr("ABC123"))
print("redacted:", redact("Booking ABC123 for john.doe@example.com is confirmed."))
print("safe_lookup('xx'):", safe_lookup("xx"))

**Principle:** validate inputs at the boundary, redact PII before it hits a log, and keep sensitive work inside your VPC with `RETURN_CONTROL`.

## Production readiness checklist

| Area | Dev (Notebooks 1–4) | Production |
|---|---|---|
| Credentials | `aws configure` key | role + short-lived creds |
| Permissions | broad / default | least-privilege, ARN-scoped |
| Throttling | crashes | adaptive retries + backoff |
| Control plane | called inline | deploy-time / CI only, ids cached |
| Observability | `print` | structured JSON logs + CloudWatch |
| Cost | unbounded | `max_turns`, `maxTokens`, model tiering |
| Releases | `TSTALIASID` (draft) | named alias on a pinned version |
| Quality | "looks right" | fixed eval set, tracked pass rate |
| Hallucination | prompt line | grounding guardrail + audit log |
| Security | none | input validation, PII redaction, in-VPC execution |

## What's next

You now have the two skills you will use every day of agentic work — **controlling the loop** and **keeping the model honest** — plus the operations layer to run them for real. The day-after builds up from here:

- **Strands Agents SDK** — the model drives the loop; less boilerplate than the hand-built version, same ReAct spine.
- **AgentCore** — runtime, memory, identity, and observability as managed building blocks (governance, not orchestration).
- **Multi-agent flows** — the verifier pattern from Notebook 4, generalised to teams of agents.

Same TravelMind use cases throughout. The frameworks change; the controls you learned here do not.